<a href="https://colab.research.google.com/github/Freddykaseya/groupe-03-IADS/blob/main/notebooks/Credit_Card_Fraud_Detection_Group03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Credit Card Fraud Detection

## Group 03

### Research Objective
Detect fraudulent credit card transactions using
machine learning algorithms.

### Models
- Logistic Regression
- Random Forest
- Isolation Forest

### Evaluation Metrics
- Precision
- Recall
- F1-score
- PR-AUC

In [2]:
# ==========================================
# 1. IMPORTATION DES BIBLIOTHÈQUES
# ==========================================

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import sklearn

from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

# Reproductibilité
RANDOM_STATE = 42

# Configuration d'affichage
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

sns.set_theme(style="whitegrid")

print("Environnement initialisé avec succès.")
print("Pandas :", pd.__version__)
print("NumPy :", np.__version__)
print("Scikit-learn :", sklearn.__version__)

Environnement initialisé avec succès.
Pandas : 2.2.3
NumPy : 2.4.6
Scikit-learn : 1.6.1


In [3]:
# Installation  kaggle
%pip install -q -U kagglehub

In [4]:
import os
import kagglehub
from google.colab import userdata
from pathlib import Path

# Récupération sécurisée du token
os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")

# Identifiant Kaggle du dataset
DATASET_ID = "mlg-ulb/creditcardfraud"

# Téléchargement
dataset_path = kagglehub.dataset_download(
    DATASET_ID,
    output_dir="/content/credit_card_fraud"
)

print("Dataset téléchargé :", dataset_path)

Using Colab cache for faster access to the 'creditcardfraud' dataset.
Dataset téléchargé : /kaggle/input/creditcardfraud


In [5]:
# Rechercher le fichier CSV téléchargé
csv_files = list(Path(dataset_path).rglob("creditcard.csv"))

if not csv_files:
    raise FileNotFoundError("creditcard.csv introuvable.")

DATA_PATH = csv_files[0]

# Chargement
df = pd.read_csv(DATA_PATH)

print("Dataset chargé avec succès.")
print("Dimensions :", df.shape)

Dataset chargé avec succès.
Dimensions : (284807, 31)


# A savoir sur  les  données  

Les  valeurs  écrites  dans ce tableau répresente de transaction, qui sont   représentées  par  des caractéristiques obtenues par une transformation mathématique appelée PCA (Principal Component Analysis), ou analyse en composantes principales.
> Ajouter une citation



In [14]:
df.head()


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,0.090794,-0.551600,-0.617801,-0.991390,-0.311169,1.468177,-0.470401,0.207971,0.025791,0.403993,0.251412,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,-0.166974,1.612727,1.065235,0.489095,-0.143772,0.635558,0.463917,-0.114805,-0.183361,-0.145783,-0.069083,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,0.207643,0.624501,0.066084,0.717293,-0.165946,2.345865,-2.890083,1.109969,-0.121359,-2.261857,0.524980,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,-0.054952,-0.226487,0.178228,0.507757,-0.287924,-0.631418,-1.059647,-0.684093,1.965775,-1.232622,-0.208038,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,0.753074,-0.822843,0.538196,1.345852,-1.119670,0.175121,-0.451449,-0.237033,-0.038195,0.803487,0.408542,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [7]:
# ==========================================
# 2. PREMIER DIAGNOSTIC DU DATASET
# ==========================================

print("DIMENSIONS DU DATASET")
print(df.shape)

print("\nNOMS DES COLONNES")
print(df.columns.tolist())

print("\nTYPES DES VARIABLES")
print(df.dtypes)

print("\nINFORMATIONS GÉNÉRALES")
df.info()

print("\nSTATISTIQUES DESCRIPTIVES")
display(df.describe())

print("\nDISTRIBUTION DE LA VARIABLE CIBLE")
display(df["Class"].value_counts())

print("\nPROPORTION DE CHAQUE CLASSE")
display(df["Class"].value_counts(normalize=True) * 100)

DIMENSIONS DU DATASET
(284807, 31)

NOMS DES COLONNES
['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'Class']

TYPES DES VARIABLES
Time      float64
V1        float64
V2        float64
V3        float64
V4        float64
V5        float64
V6        float64
V7        float64
V8        float64
V9        float64
V10       float64
V11       float64
V12       float64
V13       float64
V14       float64
V15       float64
V16       float64
V17       float64
V18       float64
V19       float64
V20       float64
V21       float64
V22       float64
V23       float64
V24       float64
V25       float64
V26       float64
V27       float64
V28       float64
Amount    float64
Class       int64
dtype: object

INFORMATIONS GÉNÉRALES
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 284807 entries, 0 to 284806
Data columns (total 31 columns):
 # 

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
count,284807.000000,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,284807.000000,284807.000000
mean,94813.859575,1.175161e-15,3.384974e-16,-1.379537e-15,2.094852e-15,1.021879e-15,1.494498e-15,-5.620335e-16,1.149614e-16,-2.414189e-15,2.238554e-15,1.724421e-15,-1.245415e-15,8.238900e-16,1.213481e-15,4.866699e-15,1.436219e-15,-3.768179e-16,9.707851e-16,1.036249e-15,6.418678e-16,1.628620e-16,-3.576577e-16,2.618565e-16,4.473914e-15,5.109395e-16,1.686100e-15,-3.661401e-16,-1.227452e-16,88.349619,0.001727
std,47488.145955,1.958696e+00,1.651309e+00,1.516255e+00,1.415869e+00,1.380247e+00,1.332271e+00,1.237094e+00,1.194353e+00,1.098632e+00,1.088850e+00,1.020713e+00,9.992014e-01,9.952742e-01,9.585956e-01,9.153160e-01,8.762529e-01,8.493371e-01,8.381762e-01,8.140405e-01,7.709250e-01,7.345240e-01,7.257016e-01,6.244603e-01,6.056471e-01,5.212781e-01,4.822270e-01,4.036325e-01,3.300833e-01,250.120109,0.041527
min,0.000000,-5.640751e+01,-7.271573e+01,-4.832559e+01,-5.683171e+00,-1.137433e+02,-2.616051e+01,-4.355724e+01,-7.321672e+01,-1.343407e+01,-2.458826e+01,-4.797473e+00,-1.868371e+01,-5.791881e+00,-1.921433e+01,-4.498945e+00,-1.412985e+01,-2.516280e+01,-9.498746e+00,-7.213527e+00,-5.449772e+01,-3.483038e+01,-1.093314e+01,-4.480774e+01,-2.836627e+00,-1.029540e+01,-2.604551e+00,-2.256568e+01,-1.543008e+01,0.000000,0.000000
25%,54201.500000,-9.203734e-01,-5.985499e-01,-8.903648e-01,-8.486401e-01,-6.915971e-01,-7.682956e-01,-5.540759e-01,-2.086297e-01,-6.430976e-01,-5.354257e-01,-7.624942e-01,-4.055715e-01,-6.485393e-01,-4.255740e-01,-5.828843e-01,-4.680368e-01,-4.837483e-01,-4.988498e-01,-4.562989e-01,-2.117214e-01,-2.283949e-01,-5.423504e-01,-1.618463e-01,-3.545861e-01,-3.171451e-01,-3.269839e-01,-7.083953e-02,-5.295979e-02,5.600000,0.000000
50%,84692.000000,1.810880e-02,6.548556e-02,1.798463e-01,-1.984653e-02,-5.433583e-02,-2.741871e-01,4.010308e-02,2.235804e-02,-5.142873e-02,-9.291738e-02,-3.275735e-02,1.400326e-01,-1.356806e-02,5.060132e-02,4.807155e-02,6.641332e-02,-6.567575e-02,-3.636312e-03,3.734823e-03,-6.248109e-02,-2.945017e-02,6.781943e-03,-1.119293e-02,4.097606e-02,1.659350e-02,-5.213911e-02,1.342146e-03,1.124383e-02,22.000000,0.000000
75%,139320.500000,1.315642e+00,8.037239e-01,1.027196e+00,7.433413e-01,6.119264e-01,3.985649e-01,5.704361e-01,3.273459e-01,5.971390e-01,4.539234e-01,7.395934e-01,6.182380e-01,6.625050e-01,4.931498e-01,6.488208e-01,5.232963e-01,3.996750e-01,5.008067e-01,4.589494e-01,1.330408e-01,1.863772e-01,5.285536e-01,1.476421e-01,4.395266e-01,3.507156e-01,2.409522e-01,9.104512e-02,7.827995e-02,77.165000,0.000000
max,172792.000000,2.454930e+00,2.205773e+01,9.382558e+00,1.687534e+01,3.480167e+01,7.330163e+01,1.205895e+02,2.000721e+01,1.559499e+01,2.374514e+01,1.201891e+01,7.848392e+00,7.126883e+00,1.052677e+01,8.877742e+00,1.731511e+01,9.253526e+00,5.041069e+00,5.591971e+00,3.942090e+01,2.720284e+01,1.050309e+01,2.252841e+01,4.584549e+00,7.519589e+00,3.517346e+00,3.161220e+01,3.384781e+01,25691.160000,1.000000



DISTRIBUTION DE LA VARIABLE CIBLE


,count
Class,
0,284315
1,492



PROPORTION DE CHAQUE CLASSE


,proportion
Class,
0,99.827251
1,0.172749


## vérification de la qualité de la DATA

In [8]:
# ==========================================
# 3. DIAGNOSTIC DE LA QUALITÉ DES DONNÉES
# ==========================================

# 1. Valeurs manquantes
print("VALEURS MANQUANTES")
missing_values = df.isnull().sum()

display(
    missing_values[missing_values > 0]
)

print(
    "Total des valeurs manquantes :",
    df.isnull().sum().sum()
)


# 2. Doublons
print("\nDOUBLONS")

duplicate_count = df.duplicated().sum()

print("Nombre de doublons :", duplicate_count)


# 3. Nombre de valeurs uniques par colonne
print("\nVALEURS UNIQUES")

unique_values = df.nunique().sort_values(
    ascending=False
)

display(unique_values)


# 4. Vérification de la variable cible
print("\nVALIDATION DE LA CIBLE")

print("Valeurs présentes :", df["Class"].unique())

print("Nombre de classes :", df["Class"].nunique())


# 5. Vérification des valeurs infinies
print("\nVALEURS INFINIES")

infinite_count = np.isinf(
    df.select_dtypes(include=np.number)
).sum().sum()

print("Nombre de valeurs infinies :", infinite_count)

VALEURS MANQUANTES


,0


Total des valeurs manquantes : 0

DOUBLONS
Nombre de doublons : 1081

VALEURS UNIQUES


,0
V1,275663
V2,275663
V3,275663
V5,275663
V4,275663
V6,275663
V7,275663
V11,275663
V8,275663
V9,275663



VALIDATION DE LA CIBLE
Valeurs présentes : [0 1]
Nombre de classes : 2

VALEURS INFINIES
Nombre de valeurs infinies : 0


## 3. Diagnostic initial de la qualité des données

Cette première analyse vise à identifier les éventuels problèmes de qualité présents dans notre jeu de données avant toute opération de nettoyage.

### 3.1 Valeurs manquantes

Aucune valeur manquante n'a été détectée dans les 31 colonnes du dataset.

**Décision :** aucune opération de remplacement ou de suppression liée aux valeurs manquantes n'est nécessaire à ce stade.

### 3.2 Doublons

L'analyse initiale a permis de détecter 1 081 lignes dupliquées.

**Décision :** aucune suppression n'est effectuée pour le moment. Une analyse complémentaire est nécessaire afin d'étudier leur répartition entre les transactions normales et frauduleuses.

### 3.3 Valeurs infinies

Aucune valeur infinie n'a été détectée dans les variables numériques.

**Décision :** aucune correction n'est nécessaire pour ce problème.

### 3.4 Variable cible

La variable cible `Class` contient deux catégories :

- 0 : transaction normale.
- 1 : transaction frauduleuse.

Le dataset présente un fort déséquilibre entre ces deux catégories, les transactions frauduleuses représentant environ 0,173 % des observations.

### 3.5 Conclusion provisoire

Le dataset ne présente aucune valeur manquante ni valeur infinie. Toutefois, la présence de 1 081 doublons nécessite une investigation complémentaire avant de décider d'une éventuelle suppression.

Le déséquilibre important de la variable cible constitue également un élément central à prendre en compte dans le choix des méthodes et des métriques d'évaluation.

In [9]:
# ==========================================
# 5. ANALYSE APPROFONDIE DES DOUBLONS
# ==========================================

# Identifier toutes les lignes appartenant
# à un groupe de doublons
duplicate_rows = df[
    df.duplicated(keep=False)
].copy()

print("Nombre de lignes appartenant à des groupes de doublons :",
      len(duplicate_rows))

# Répartition des lignes dupliquées selon Class
print("\nRépartition des doublons selon la cible :")

display(
    duplicate_rows["Class"].value_counts()
)

# Pourcentages
print("\nPourcentage par classe :")

display(
    duplicate_rows["Class"]
    .value_counts(normalize=True) * 100
)

# Comparaison avec le dataset complet
print("\nRépartition de la cible dans le dataset complet :")

display(
    df["Class"].value_counts()
)

Nombre de lignes appartenant à des groupes de doublons : 1854

Répartition des doublons selon la cible :


,count
Class,
0,1822
1,32



Pourcentage par classe :


,proportion
Class,
0,98.274002
1,1.725998



Répartition de la cible dans le dataset complet :


,count
Class,
0,284315
1,492


In [10]:
# Identifier les lignes appartenant à des groupes de doublons
duplicate_rows = df[df.duplicated(keep=False)].copy()

# Compter le nombre d'occurrences de chaque ligne identique
duplicate_groups = (
    duplicate_rows
    .groupby(list(df.columns), dropna=False)
    .size()
    .reset_index(name="nombre_occurrences")
)

print("Nombre de groupes de doublons :", len(duplicate_groups))

print("\nDistribution du nombre d'occurrences :")
display(duplicate_groups["nombre_occurrences"].value_counts().sort_index())

print("\nNombre de groupes de doublons par classe :")
display(duplicate_groups.groupby("Class").size())

print("\nExemples de groupes de doublons :")
display(duplicate_groups.head(10))

Nombre de groupes de doublons : 773

Distribution du nombre d'occurrences :


,count
nombre_occurrences,
2,611
3,66
4,81
5,10
6,1
9,2
18,2



Nombre de groupes de doublons par classe :


,0
Class,
0,760
1,13



Exemples de groupes de doublons :


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class,nombre_occurrences
0,26.0,-0.535388,0.865268,1.351076,0.147575,0.433680,0.086983,0.693039,0.179742,-0.285642,-0.482474,0.871800,0.853447,-0.571822,0.102252,-1.519991,-0.285912,-0.309633,-0.403902,-0.823743,-0.283264,0.049526,0.206537,-0.187108,0.000753,0.098117,-0.553471,-0.078306,0.025427,1.77,0,2
1,26.0,-0.529912,0.873892,1.347247,0.145457,0.414209,0.100223,0.711206,0.176066,-0.286717,-0.484688,0.872490,0.851636,-0.571745,0.100974,-1.519772,-0.284376,-0.310524,-0.404248,-0.823374,-0.290348,0.046949,0.208105,-0.185548,0.001031,0.098816,-0.552904,-0.073288,0.023307,6.14,0,2
2,74.0,1.038370,0.127486,0.184456,1.109950,0.441699,0.945283,-0.036715,0.350995,0.118950,-0.243289,0.578063,0.674730,-0.534231,0.446601,1.122885,-1.768001,1.241157,-2.449500,-1.747255,-0.335520,0.102520,0.605089,0.023092,-0.626463,0.479120,-0.166937,0.081247,0.001192,1.18,0,4
3,145.0,-2.420413,1.947885,0.553646,0.983069,-0.281518,2.408958,-1.401613,-0.188299,0.675878,0.158497,-1.224951,1.314705,0.304412,-0.290218,-2.014548,-0.772801,0.696761,0.017914,2.177250,-0.276207,1.213826,-1.238620,0.006927,-1.724222,0.239603,-0.313703,-0.188281,0.119831,6.00,0,2
4,145.0,-2.419486,1.949346,0.552998,0.982710,-0.284815,2.411200,-1.398537,-0.188922,0.675695,0.158123,-1.224834,1.314398,0.304425,-0.290434,-2.014511,-0.772540,0.696610,0.017855,2.177312,-0.277406,1.213390,-1.238354,0.007191,-1.724175,0.239721,-0.313607,-0.187431,0.119472,6.74,0,2
5,919.0,0.904289,-0.538055,0.396058,0.500680,-0.864473,-0.657199,0.027231,-0.029473,0.265447,-0.110991,1.254250,0.704631,-0.967461,0.466524,-0.303979,0.094702,-0.198326,-0.124616,0.401377,0.157346,-0.099460,-0.597579,-0.048666,0.551824,0.182934,0.402176,-0.081357,0.027252,158.00,0,2
6,919.0,1.207596,-0.036860,0.572104,0.373148,-0.709633,-0.713698,-0.181105,0.011277,0.283940,-0.037866,1.252000,0.698164,-0.995452,0.430980,-0.323729,0.073995,-0.202981,-0.156783,0.451522,-0.162378,-0.194591,-0.514717,0.089714,0.543768,0.240581,0.418921,-0.051693,-0.000085,1.00,0,2
7,1074.0,-2.445204,-0.505494,0.164511,-0.449658,0.460784,-0.509035,-0.490478,1.060597,-0.385591,-1.053488,-0.553817,0.207330,-0.933946,0.969621,-0.758698,0.863809,-0.533855,0.559857,0.502850,0.171937,0.060361,-0.802101,-0.299229,-0.831608,-0.055518,0.315454,-0.467257,-0.167591,1.00,0,2
8,1074.0,-2.248488,-0.195670,0.026949,-0.525780,-0.238759,-0.033359,0.162196,0.928520,-0.424215,-1.133001,-0.529027,0.142246,-0.931195,0.923712,-0.750816,0.919012,-0.565837,0.547431,0.516124,-0.082562,-0.032208,-0.745757,-0.243193,-0.821633,-0.030402,0.335838,-0.286998,-0.243768,158.00,0,2
9,1308.0,-1.379848,0.536720,2.647870,1.408111,0.126442,0.986525,0.960251,-0.629717,1.729173,1.346832,-0.622458,0.516210,0.174168,-2.023273,-1.185043,-2.036791,0.653358,-1.294348,1.297445,0.464675,-0.666874,-0.343574,0.022138,0.093532,0.011293,-0.503395,-0.438189,-0.346731,43.80,0,2


In [11]:
# Examiner uniquement les groupes de doublons frauduleux

fraud_duplicate_groups = duplicate_groups[
    duplicate_groups["Class"] == 1
].copy()

print("Nombre de groupes de doublons frauduleux :",
      len(fraud_duplicate_groups))

print("\nNombre total d'occurrences dans ces groupes :",
      fraud_duplicate_groups["nombre_occurrences"].sum())

print("\nDistribution de leurs occurrences :")
display(
    fraud_duplicate_groups["nombre_occurrences"]
    .value_counts()
    .sort_index()
)

print("\nDétail des groupes frauduleux :")
display(fraud_duplicate_groups)

Nombre de groupes de doublons frauduleux : 13

Nombre total d'occurrences dans ces groupes : 32

Distribution de leurs occurrences :


,count
nombre_occurrences,
2,11
4,1
6,1



Détail des groupes frauduleux :


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class,nombre_occurrences
281,68207.0,-13.192671,12.785971,-9.906650,3.320337,-4.801176,5.760059,-18.750889,-37.353443,-0.391540,-5.052502,4.406806,-4.610756,-1.909488,-9.072711,-0.226074,-6.211557,-6.248145,-3.149247,0.051576,-3.493050,27.202839,-8.887017,5.303607,-0.639435,0.263203,-0.108877,1.269566,0.939407,1.00,1,6
393,84204.0,-1.927453,1.827621,-7.019495,5.348303,-2.739188,-2.107219,-5.015848,1.205868,-4.382713,-8.337707,7.190306,-9.424844,-0.223293,-12.875494,-0.071918,-6.299961,-12.719207,-3.740176,0.844060,2.172709,1.376938,-0.792017,-0.771414,-0.379574,0.718717,1.111151,1.277707,0.819081,512.25,1,2
394,84204.0,-0.937843,3.462889,-6.445104,4.932199,-2.233983,-2.291561,-5.695594,1.338825,-4.322377,-8.099119,7.182967,-9.445943,-0.314620,-12.991466,-0.136359,-6.367524,-12.734394,-3.845130,1.007667,1.129532,1.066550,-0.521657,-0.319917,-0.405859,0.906802,1.165784,1.374495,0.729889,0.00,1,2
398,85285.0,-7.030308,3.421991,-9.525072,5.270891,-4.024630,-2.865682,-6.989195,3.791551,-4.622730,-8.409665,6.309044,-8.576761,0.246747,-11.534046,-0.364265,-5.452495,-11.887570,-3.563585,0.876019,0.545698,1.103398,-0.541855,0.036943,-0.355519,0.353634,1.042458,1.359516,-0.272188,0.00,1,2
399,85285.0,-6.713407,3.921104,-9.746678,5.148263,-5.151563,-2.099389,-5.937767,3.578780,-4.684952,-8.537758,6.348979,-8.681609,0.251179,-11.608002,-0.351569,-5.363566,-11.939092,-3.583603,0.897402,0.135711,0.954272,-0.451086,0.127214,-0.339450,0.394096,1.075295,1.649906,-0.394905,252.92,1,2
429,93853.0,-6.185857,7.102985,-13.030455,8.010823,-7.885237,-3.974550,-12.229608,4.971232,-4.248307,-12.965481,8.688308,-17.182918,0.069577,-14.116156,0.959032,-12.375334,-18.716765,-6.522015,3.517955,0.483930,2.502772,0.481691,0.480958,0.360319,-0.293354,-0.199193,-0.203917,0.398927,44.90,1,2
430,93853.0,-5.839192,7.151532,-12.816760,7.031115,-9.651272,-2.938427,-11.543207,4.843627,-3.494276,-13.320789,8.460244,-17.003289,0.101557,-14.094452,0.747031,-12.661696,-18.912494,-6.626975,4.008921,0.055684,2.462056,1.054865,0.530481,0.472670,-0.275998,0.282435,0.104886,0.254417,316.06,1,2
431,93860.0,-10.850282,6.727466,-16.760583,8.425832,-10.252697,-4.192171,-14.077086,7.168288,-3.683242,-15.239962,8.030708,-16.060306,0.270530,-14.952981,-0.241095,-11.866731,-15.486990,-5.748652,4.130031,-0.646818,2.541637,0.135535,-1.023967,0.406265,0.106593,-0.026232,-1.464630,-0.411682,78.00,1,2
432,93860.0,-10.632375,7.251936,-17.681072,8.204144,-10.166591,-4.510344,-12.981606,6.783589,-4.659330,-14.924655,8.389142,-16.465504,0.338517,-14.224404,0.556584,-11.683998,-15.841616,-5.753200,3.813041,-0.810146,2.715357,0.695603,-1.138122,0.459442,0.386337,0.522438,-1.416604,-0.488307,188.52,1,2
433,93879.0,-13.086519,7.352148,-18.256576,10.648505,-11.731476,-3.659167,-14.873658,8.810473,-5.418204,-13.202577,6.357227,-15.531611,0.659695,-11.412330,-2.447576,-9.833743,-18.174617,-7.269905,0.623797,-1.376298,2.761157,-0.266162,-0.412861,0.519952,-0.743909,-0.167808,-2.498300,-0.711066,30.31,1,2


In [12]:
# Conserver une copie du dataset original
df_original = df.copy()

# Supprimer les doublons exacts en conservant la première occurrence
df_clean = df.drop_duplicates(keep="first").copy()

# Comparer les dimensions avant et après nettoyage
print("AVANT NETTOYAGE :", df_original.shape)
print("APRÈS NETTOYAGE :", df_clean.shape)

print("\nNombre de lignes supprimées :",
      len(df_original) - len(df_clean))

print("\nRépartition des classes avant nettoyage :")
display(df_original["Class"].value_counts())

print("\nRépartition des classes après nettoyage :")
display(df_clean["Class"].value_counts())

print("\nVérification des doublons restants :",
      df_clean.duplicated().sum())

AVANT NETTOYAGE : (284807, 31)
APRÈS NETTOYAGE : (283726, 31)

Nombre de lignes supprimées : 1081

Répartition des classes avant nettoyage :


,count
Class,
0,284315
1,492



Répartition des classes après nettoyage :


,count
Class,
0,283253
1,473



Vérification des doublons restants : 0


## 4. Traitement des doublons

L'analyse de la qualité des données a révélé 1 081 occurrences supplémentaires
correspondant à des lignes strictement identiques.

L'investigation a permis d'identifier 773 groupes de doublons, dont 760 groupes
associés aux transactions normales et 13 groupes associés aux transactions
frauduleuses.

Après examen, nous avons décidé de supprimer les répétitions exactes en
conservant une seule occurrence de chaque observation.

Cette décision vise à éviter que des observations identiques soient présentes
dans plusieurs ensembles de données et puissent fausser l'évaluation des modèles.

Le dataset original a été conservé dans `df_original`, tandis que le dataset
nettoyé est stocké dans `df_clean`.

Les effectifs avant et après nettoyage sont enregistrés afin d'assurer la
traçabilité de cette opération.